# NB6 -- Attack 3: Adaptive White-Box Attack vs. Defended Model (Multiclass, v4)

Rebuilt against v4 from the leakage-era `NB6_Attack3_Drive_results.ipynb`
reference. The attacker retrains a surrogate to mimic the **defended**
model's decision boundary (not the standard model's), then PGD-attacks the
defended model with that adaptive surrogate across an epsilon sweep.

**Corrections applied vs. the reference:**
- Paths/loading: local Windows, `phase1_data_v4.pkl`, GPU-detection pattern.
- Standard model: retrained fresh from the real v4 `best_params` (no fitted
  tuned-booster artifact exists), same recipe as NB3/NB4/NB5.
- Defended model: **loaded directly** from `xgb_defended_model_v4.json`
  (Defence 1's actual artifact) -- cannot be retrained/reproduced without
  redoing all of Defence 1's specific augmented training set.
- **Fixed the flat-line comparison assumption**: the reference notebook
  assumed "Robust IDS + Attack 1 PGD" evasion is 0.0 at *every* epsilon in the
  sweep, extrapolated from the single eps=0.1 point Defence 1 actually
  measured (`atk1_def_f1 = [0.9985] * len(EPSILONS)`). No data backs the other
  10 points. This version instead **loads Attack 1's actual saved surrogate**
  (`surrogate_mlp_v4.pt`) and runs a genuine full epsilon sweep of the
  *original* (non-adaptive) PGD attack against the *defended* model, so all
  three comparison lines in the final plot are real, not two real points and
  a fabricated flat line.
- Hardcoded leakage-era summary numbers (clean F1 0.9987/0.9985, PPO evasion
  0.7440/0.0060) replaced with real v4 values pulled from
  `defence1_results_v4.pkl`.
- **Seed-robustness built in from the start** (5 seeds: 42, 7, 123, 2024, 99),
  not retrofitted afterward. The adaptive surrogate is the only stochastic
  component here (weight init, batch shuffling, dropout) -- PGD generation
  itself is deterministic given a fixed surrogate. This is cheap to do
  properly since PGD is fully vectorized/batched, unlike the PPO rollouts in
  Attack 2/Defence 1, so there's no subsampling tradeoff needed: every seed
  gets the full epsilon sweep on the full test set.
- No binary counterpart existed in the reference; built separately as a
  companion notebook.


---

**This revision (Attack 3, baseline-model variant):** the original NB6 attacks the Optuna-tuned XGBoost's defended model, matching Attack 1/2's original target. This revision retargets the identical adaptive-attack methodology (adaptive surrogate training, 5-seed epsilon sweep, three-way comparison) at the untuned baseline pipeline instead. Five cross-referenced files were fixed to point at the baseline artifacts: the Standard model now uses fixed baseline hyperparameters instead of loading `optuna_xgboost_results_v4.pkl`; the defended model loads `xgb_defended_model_baseline_v4.json` (from the Defence 1 baseline notebook); and the Attack 1 comparison data, Defence 1 comparison data, and PGD adversarial examples all load their `*_baseline_v4` equivalents. Output filenames are suffixed `_baseline_v4` throughout so this run doesn't overwrite the original tuned-model results. **Dependency: requires `xgb_defended_model_baseline_v4.json`, `attack1_results_baseline_v4.pkl`, `defence1_results_baseline_v4.pkl`, and `X_adv_pgd_eps01_baseline_v4.npy` to already exist on disk** -- i.e. the Attack 1 and Defence 1 baseline notebooks must be run first.

---

**Further fixed (Drive-ready revision):** the non-adaptive baseline sweep was still loading `surrogate_mlp_v4.pt` -- the *tuned* pipeline's Attack 1 surrogate -- a sixth cross-contamination the earlier pass missed. Now loads `surrogate_mlp_baseline_v4.pt` (saved by the Attack 1 baseline notebook). Outputs were stripped; run fresh.


## Setup

In [ ]:
import os, random, warnings, copy
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Drive-ready: PHYSAR-NEW lives on Google Drive (survives Colab runtime restarts)
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/PHYSAR-NEW'
os.chdir(PROJECT_DIR)
if not os.path.exists('phase1_data_v4.pkl'):
    print(f'WARNING: phase1_data_v4.pkl not found in {PROJECT_DIR} -- fix PROJECT_DIR before continuing.')
    print(sorted(os.listdir(PROJECT_DIR))[:20])
else:
    print('Found phase1_data_v4.pkl -- PROJECT_DIR looks correct.')

RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Working directory: {os.getcwd()}')
print(f'Device: {DEVICE}')

with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2)
    _test_model.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _fell_back_to_cpu = any('No visible GPU' in str(w.message) for w in _caught)

if _fell_back_to_cpu:
    GPU_PARAMS = {}
    print('GPU not available for XGBoost -- using CPU.')
else:
    GPU_PARAMS = {'tree_method': 'hist', 'device': 'cuda'}
    print('GPU detected -- XGBoost will use CUDA.')


## 1. Load v4 artifacts: data, standard model, defended model, Attack 1 + Defence 1 results

In [ ]:
PIPELINE_VERSION = 'v4'
EXPECTED_TEST_FRAC = 0.15
EXPECTED_EMBARGO_SECONDS = 1.0

data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
version_mismatch = (
    data.get('PIPELINE_VERSION') != PIPELINE_VERSION
    or data.get('TEST_FRAC') != EXPECTED_TEST_FRAC
    or data.get('EMBARGO_SECONDS') != EXPECTED_EMBARGO_SECONDS
)
if version_mismatch:
    print('WARNING: version metadata mismatch -- proceeding anyway, double check first.')
else:
    print(f'Version check passed: {PIPELINE_VERSION}')

X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train = pd.Series(data['y_train'])
y_test  = pd.Series(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']
y_train_np = y_train.values
y_test_np  = y_test.values
n_feat = X_train_scaled.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
attack_mask_test = (y_test_np != 0)

# CHANGED: this notebook now targets the UNTUNED baseline XGBoost (Table 6.1's
# headline model, F1=0.7225) rather than the Optuna-tuned configuration (F1=0.7110)
# the original NB6 used -- companion to the Attack 1/2 and Defence 1 baseline checks.
baseline_results = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')
best_params = dict(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=6, eval_metric='mlogloss',
    random_state=RANDOM_STATE, n_jobs=-1, **GPU_PARAMS,
)
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train_scaled, y_train, verbose=False)
standard_test_f1 = f1_score(y_test, xgb_model.predict(X_test_scaled), average='weighted')
print(f'Standard XGBoost (untuned baseline) -- Test F1: {standard_test_f1:.4f} '
      f'(recorded: {baseline_results["xgb_test_f1"]:.4f})')

# --- Defended model: LOADED directly, not retrained ---
# Loaded via the native Booster API + a thin wrapper, not xgb.XGBClassifier().load_model()
# -- a freshly-constructed XGBClassifier that was never .fit() doesn't get its sklearn
# wrapper attributes (n_classes_, classes_) populated by load_model() alone, and in
# recent xgboost versions classes_ is a read-only derived property (can't just set it
# either), so .predict() raises AttributeError. The Booster itself loads and predicts
# fine -- this only works around the sklearn wrapper's bookkeeping requirement.
defended_booster = xgb.Booster()
defended_booster.load_model(f'xgb_defended_model_baseline_{PIPELINE_VERSION}.json')

def defended_predict(X):
    probs = defended_booster.predict(xgb.DMatrix(X))
    return probs.argmax(axis=1)

class DefendedModelWrapper:
    def predict(self, X):
        return defended_predict(X)

xgb_defended = DefendedModelWrapper()
defended_test_f1 = f1_score(y_test, xgb_defended.predict(X_test_scaled), average='weighted')
print(f'Defended XGBoost (loaded) -- Test F1: {defended_test_f1:.4f}')

# CHANGED: cross-reference the baseline-model Attack 1/Defence 1 runs, not the
# tuned-model ones, for an apples-to-apples comparison throughout this notebook.
attack1_data = joblib.load(f'attack1_results_baseline_{PIPELINE_VERSION}.pkl')
results_pgd_atk1 = attack1_data['results_pgd']
defence1_data = joblib.load(f'defence1_results_baseline_{PIPELINE_VERSION}.pkl')
X_adv_pgd_eps01 = np.load(f'X_adv_pgd_eps01_baseline_{PIPELINE_VERSION}.npy')

print(f'Attack 1 results loaded: {len(results_pgd_atk1)} epsilon points')
print(f"Defence 1 results loaded: clean F1 defended={defence1_data['clean_test_f1_defended']:.4f}, "
      f"PGD evasion defended={defence1_data['pgd_evasion_defended']:.4f}")


## 2. Honest non-adaptive baseline: Attack 1's original surrogate vs. the defended model

Loads Attack 1's actual trained surrogate and runs a genuine full epsilon
sweep against the defended model, replacing the reference notebook's
fabricated flat-line assumption. This is the fair "did the attacker even need
to adapt?" baseline.

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

original_surrogate = SurrogateMLP(n_feat).to(DEVICE)
original_surrogate.load_state_dict(torch.load(f'surrogate_mlp_baseline_{PIPELINE_VERSION}.pt', map_location=DEVICE))
original_surrogate.eval()
print("Loaded Attack 1's original surrogate.")

def pgd_generate(model, X, y_np, epsilon, alpha=None, n_iter=10):
    model.eval()
    if alpha is None:
        alpha = epsilon / 4
    y_t_ = torch.from_numpy(y_np).long().to(DEVICE)
    X_orig = torch.from_numpy(X).float().to(DEVICE)
    X_adv = X_orig.clone().detach()
    for _ in range(n_iter):
        X_adv.requires_grad_(True)
        loss = F.cross_entropy(model(X_adv), y_t_)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv.detach() + alpha * grad.sign()
        perturbation = torch.clamp(X_adv - X_orig, -epsilon, epsilon)
        X_adv = (X_orig + perturbation).detach()
    return X_adv.cpu().numpy()

EPSILONS = [0.0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
BATCH = 4096

def epsilon_sweep(surrogate_model, target_model, epsilons=EPSILONS):
    results = []
    for eps in epsilons:
        if eps == 0.0:
            X_adv = X_test_scaled.copy()
        else:
            X_adv = np.zeros_like(X_test_scaled)
            for start in range(0, len(X_test_scaled), BATCH):
                end = min(start + BATCH, len(X_test_scaled))
                X_adv[start:end] = pgd_generate(surrogate_model, X_test_scaled[start:end], y_test_np[start:end], eps)
        pred = target_model.predict(X_adv)
        overall_f1 = f1_score(y_test_np, pred, average='weighted')
        evasion = (pred[attack_mask_test] == 0).mean()
        results.append({'epsilon': eps, 'overall_f1': overall_f1, 'evasion_rate': evasion})
    return results

print('Running honest non-adaptive baseline sweep (original surrogate vs. defended model)...')
results_nonadaptive_vs_defended = epsilon_sweep(original_surrogate, xgb_defended)
for r in results_nonadaptive_vs_defended:
    print(f"  eps={r['epsilon']:.2f}  F1={r['overall_f1']:.4f}  Evasion={r['evasion_rate']:.4f}")


## 3. Adaptive surrogate architecture and training function

**Updated again after the capacity/epoch increase made no difference**
(mean agreement 0.6987 vs. the original 0.700 -- within noise of each other).
The per-epoch printout from that run showed *why*: training loss kept
falling while test-set agreement peaked at epoch 1 (0.7331) and drifted down
from there -- overfitting to train-set-specific quirks of the defended
model's predictions, not undertraining. This version checks agreement every
epoch and **keeps the best checkpoint** (early stopping on peak held-out
agreement) instead of the final-epoch weights, directly targeting that
failure mode rather than adding more capacity that isn't the bottleneck.

In [ ]:
class AdaptiveSurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

print('Getting defended model predictions on training data (adaptive surrogate training target)...')
defended_train_pred = xgb_defended.predict(X_train_scaled)
defended_test_pred_fixed = xgb_defended.predict(X_test_scaled)  # computed once, reused every epoch check

def train_adaptive_surrogate(seed, n_epochs=100, verbose=False):
    # Early stopping on peak TEST-set agreement: the previous run showed agreement
    # peaking at epoch 1 (0.7331) then drifting down to 0.6881 by epoch 100 even as
    # training loss kept falling -- classic overfitting to train-set-specific quirks
    # of the defended model's predictions, not a capacity/undertraining problem.
    # Checking agreement every epoch and keeping the best checkpoint (rather than the
    # final-epoch weights) directly targets that failure mode instead of just adding
    # more capacity/epochs, which didn't help last time.
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AdaptiveSurrogateMLP(n_feat).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()
    X_t = torch.from_numpy(X_train_scaled).float()
    y_def_t = torch.from_numpy(defended_train_pred).long()
    loader = DataLoader(TensorDataset(X_t, y_def_t), batch_size=256, shuffle=True,
                         generator=torch.Generator().manual_seed(seed))

    X_test_t = torch.from_numpy(X_test_scaled).float().to(DEVICE)
    best_agreement = -1.0
    best_state = None
    best_epoch = -1

    for epoch in range(n_epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item(); n_batches += 1
        scheduler.step()

        model.eval()
        with torch.no_grad():
            epoch_pred = model(X_test_t).argmax(dim=1).cpu().numpy()
        epoch_agreement = (epoch_pred == defended_test_pred_fixed).mean()
        if epoch_agreement > best_agreement:
            best_agreement = epoch_agreement
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        if verbose and ((epoch + 1) % 20 == 0 or epoch == 0):
            print(f'  Epoch {epoch+1}/{n_epochs}  Loss: {total_loss/n_batches:.4f}  '
                  f'Agreement: {epoch_agreement:.4f}  (best so far: {best_agreement:.4f} @ epoch {best_epoch})')

    model.load_state_dict(best_state)
    model.eval()
    if verbose:
        print(f'  -> Restored best checkpoint from epoch {best_epoch} (agreement {best_agreement:.4f}), '
              f'discarding final-epoch weights (epoch {n_epochs}, worse or equal).')
    return model, best_agreement


## 4. Multi-seed adaptive attack: train surrogate, run epsilon sweep, per class at eps=0.1

Built in from the start rather than added as a follow-up check -- 5 seeds,
full epsilon sweep and full test set every time (cheap here since PGD
generation is fully vectorized).

In [ ]:
SEED_LIST = [42, 7, 123, 2024, 99]
EPS_DETAIL = 0.1

seed_results = []
for run_idx, seed in enumerate(SEED_LIST):
    print(f'\n{"="*60}\nSeed {seed} ({run_idx+1}/{len(SEED_LIST)})\n{"="*60}')
    adaptive_surrogate_s, agreement_s = train_adaptive_surrogate(seed, n_epochs=100, verbose=(run_idx == 0))
    print(f'  Adaptive surrogate agreement with defended model: {agreement_s:.4f} (target: >0.95)')

    sweep_s = epsilon_sweep(adaptive_surrogate_s, xgb_defended)

    r_eps01_s = next(r for r in sweep_s if r['epsilon'] == EPS_DETAIL)
    if EPS_DETAIL == 0.0:
        X_adv_detail_s = X_test_scaled.copy()
    else:
        X_adv_detail_s = np.zeros_like(X_test_scaled)
        for start in range(0, len(X_test_scaled), BATCH):
            end = min(start + BATCH, len(X_test_scaled))
            X_adv_detail_s[start:end] = pgd_generate(adaptive_surrogate_s, X_test_scaled[start:end], y_test_np[start:end], EPS_DETAIL)
    pred_detail_s = xgb_defended.predict(X_adv_detail_s)
    per_class_s = {class_names[c]: (pred_detail_s[y_test_np == c] == 0).mean() for c in range(1, 6)}

    seed_results.append({
        'seed': seed, 'agreement': agreement_s, 'sweep': sweep_s,
        'eps01_overall_f1': r_eps01_s['overall_f1'], 'eps01_evasion': r_eps01_s['evasion_rate'],
        'per_class_eps01': per_class_s,
    })
    print(f"  eps=0.1  F1={r_eps01_s['overall_f1']:.4f}  Evasion={r_eps01_s['evasion_rate']:.4f}")

    if run_idx == 0:
        adaptive_surrogate_primary = adaptive_surrogate_s  # kept for saving/inspection


## 5. Aggregate across seeds

In [ ]:
agreements = [r['agreement'] for r in seed_results]
eps01_f1s = [r['eps01_overall_f1'] for r in seed_results]
eps01_evs = [r['eps01_evasion'] for r in seed_results]

print('=' * 70)
print('SEED-ROBUSTNESS SUMMARY (n=5 seeds)')
print('=' * 70)
print(f"Adaptive surrogate agreement with defended model:  {np.mean(agreements):.4f} +/- {np.std(agreements):.4f}")
print(f"Adaptive PGD F1 at eps=0.1 (vs. defended):          {np.mean(eps01_f1s):.4f} +/- {np.std(eps01_f1s):.4f}")
print(f"Adaptive PGD Evasion at eps=0.1 (vs. defended):     {np.mean(eps01_evs):.4f} +/- {np.std(eps01_evs):.4f}")

# Per-class mean +/- std at eps=0.1
print('\nPer-class evasion at eps=0.1 (mean +/- std across seeds):')
for cls_name in class_names[1:]:
    vals = [r['per_class_eps01'][cls_name] for r in seed_results]
    print(f"  {cls_name:<14} {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

# Full sweep mean +/- std per epsilon
sweep_by_eps = {eps: [] for eps in EPSILONS}
for r in seed_results:
    for point in r['sweep']:
        sweep_by_eps[point['epsilon']].append(point['evasion_rate'])
print('\nFull sweep, evasion mean +/- std per epsilon:')
for eps in EPSILONS:
    vals = sweep_by_eps[eps]
    print(f"  eps={eps:.2f}  {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

joblib.dump(seed_results, f'attack3_seed_results_baseline_{PIPELINE_VERSION}.pkl')


## 6. Three-way comparison plot (mean adaptive curve +/- std band)

In [ ]:
atk1_eps    = [r['epsilon']    for r in results_pgd_atk1]
atk1_std_f1 = [r['overall_f1'] for r in results_pgd_atk1]
atk1_std_ev = [r['evasion_rate'] for r in results_pgd_atk1]

nonadap_eps = [r['epsilon']    for r in results_nonadaptive_vs_defended]
nonadap_f1  = [r['overall_f1'] for r in results_nonadaptive_vs_defended]
nonadap_ev  = [r['evasion_rate'] for r in results_nonadaptive_vs_defended]

adap_f1_mean = np.array([np.mean([r['sweep'][i]['overall_f1'] for r in seed_results]) for i in range(len(EPSILONS))])
adap_f1_std  = np.array([np.std([r['sweep'][i]['overall_f1'] for r in seed_results]) for i in range(len(EPSILONS))])
adap_ev_mean = np.array([np.mean([r['sweep'][i]['evasion_rate'] for r in seed_results]) for i in range(len(EPSILONS))])
adap_ev_std  = np.array([np.std([r['sweep'][i]['evasion_rate'] for r in seed_results]) for i in range(len(EPSILONS))])
eps_arr = np.array(EPSILONS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(atk1_eps, atk1_std_f1, 'o--', color='red', label='Standard IDS + Attack 1 PGD')
axes[0].plot(nonadap_eps, nonadap_f1, 's--', color='green', label='Robust IDS + Attack 1 PGD (non-adaptive)')
axes[0].plot(eps_arr, adap_f1_mean, '^-', color='orange', label='Robust IDS + Attack 3 Adaptive PGD (mean, n=5 seeds)')
axes[0].fill_between(eps_arr, adap_f1_mean - adap_f1_std, adap_f1_mean + adap_f1_std, color='orange', alpha=0.2)
axes[0].set_xlabel('Epsilon'); axes[0].set_ylabel('Overall Test F1')
axes[0].set_title('Three-Way F1 Comparison'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3); axes[0].set_ylim([0, 1.05])

axes[1].plot(atk1_eps, atk1_std_ev, 'o--', color='red', label='Standard IDS + Attack 1 PGD')
axes[1].plot(nonadap_eps, nonadap_ev, 's--', color='green', label='Robust IDS + Attack 1 PGD (non-adaptive)')
axes[1].plot(eps_arr, adap_ev_mean, '^-', color='orange', label='Robust IDS + Attack 3 Adaptive PGD (mean, n=5 seeds)')
axes[1].fill_between(eps_arr, adap_ev_mean - adap_ev_std, adap_ev_mean + adap_ev_std, color='orange', alpha=0.2)
axes[1].set_xlabel('Epsilon'); axes[1].set_ylabel('Overall Evasion Rate')
axes[1].set_title('Three-Way Evasion Rate Comparison'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(f'attack3_three_way_comparison_baseline_{PIPELINE_VERSION}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: attack3_three_way_comparison_baseline_{PIPELINE_VERSION}.png')


## 7. Summary and interpretation

In [ ]:
print('=' * 70)
print('ATTACK 3 SUMMARY: Adaptive White-Box vs. Defended XGBoost (n=5 seeds)')
print('=' * 70)

std_eps01 = next(r for r in results_pgd_atk1 if r['epsilon'] == EPS_DETAIL)
nonadap_eps01 = next(r for r in results_nonadaptive_vs_defended if r['epsilon'] == EPS_DETAIL)

print(f'\nAt epsilon={EPS_DETAIL}:')
print(f"  Standard IDS + Attack 1 PGD:              F1={std_eps01['overall_f1']:.4f}  Evasion={std_eps01['evasion_rate']:.4f}")
print(f"  Robust IDS + Attack 1 PGD (non-adaptive):  F1={nonadap_eps01['overall_f1']:.4f}  Evasion={nonadap_eps01['evasion_rate']:.4f}")
print(f"  Robust IDS + Attack 3 Adaptive PGD (mean): F1={np.mean(eps01_f1s):.4f} +/- {np.std(eps01_f1s):.4f}  "
      f"Evasion={np.mean(eps01_evs):.4f} +/- {np.std(eps01_evs):.4f}")
print(f'\nAdaptive surrogate agreement (mean): {np.mean(agreements):.4f} +/- {np.std(agreements):.4f}')

mean_evasion = np.mean(eps01_evs)
print('\nInterpretation:')
if mean_evasion < 0.05:
    print('  -> Defence HOLDS under adaptive white-box attack (mean evasion < 5%).')
elif mean_evasion < 0.20:
    print('  -> Defence PARTIALLY holds (mean evasion 5-20%).')
else:
    print('  -> Adaptive attack SIGNIFICANTLY breaks the defence (mean evasion > 20%) --')
    print('     motivating Defence 2 (autoencoder gate) as a secondary layer.')

print(f'\n{"="*70}\nCOMPLETE PIPELINE RESULTS (Standard vs. Robust IDS)\n{"="*70}')
print(f"{'Metric':<40} {'Standard IDS':>14} {'Robust IDS':>14}")
print('-' * 70)
print(f"{'Clean Test F1':<40} {standard_test_f1:>14.4f} {defended_test_f1:>14.4f}")
print(f"{'PGD Test F1 (eps=0.1)':<40} {std_eps01['overall_f1']:>14.4f} {defence1_data['pgd_f1_defended']:>14.4f}")
print(f"{'PGD Evasion (eps=0.1)':<40} {std_eps01['evasion_rate']:>14.4f} {defence1_data['pgd_evasion_defended']:>14.4f}")
print(f"{'PPO budget-matched Evasion':<40} {defence1_data['ppo_budgeted_evasion_standard']:>14.4f} {defence1_data['ppo_budgeted_evasion_defended']:>14.4f}")
print(f"{'PPO unconstrained Evasion':<40} {defence1_data['ppo_unconstrained_evasion_standard']:>14.4f} {defence1_data['ppo_unconstrained_evasion_defended']:>14.4f}")
print(f"{'Adaptive PGD Evasion (eps=0.1, mean)':<40} {'N/A':>14} {np.mean(eps01_evs):>14.4f}")


## 8. Save Attack 3 results

In [ ]:
results_to_save = []
for r in seed_results:
    results_to_save.append({
        'seed': r['seed'], 'agreement': r['agreement'],
        'sweep': [{'epsilon': p['epsilon'], 'overall_f1': p['overall_f1'], 'evasion_rate': p['evasion_rate']} for p in r['sweep']],
        'eps01_overall_f1': r['eps01_overall_f1'], 'eps01_evasion': r['eps01_evasion'],
        'per_class_eps01': r['per_class_eps01'],
    })

joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION,
    'seed_list': SEED_LIST,
    'per_seed_results': results_to_save,
    'results_nonadaptive_vs_defended': results_nonadaptive_vs_defended,
    'mean_agreement': float(np.mean(agreements)), 'std_agreement': float(np.std(agreements)),
    'mean_eps01_evasion': float(np.mean(eps01_evs)), 'std_eps01_evasion': float(np.std(eps01_evs)),
    'mean_eps01_f1': float(np.mean(eps01_f1s)), 'std_eps01_f1': float(np.std(eps01_f1s)),
}, f'attack3_results_baseline_{PIPELINE_VERSION}.pkl')

torch.save(adaptive_surrogate_primary.state_dict(), f'adaptive_surrogate_baseline_{PIPELINE_VERSION}.pt')
print('(Only the seed=42 adaptive surrogate is saved as a representative artifact; '
      'all 5 seeds\' metrics are saved in attack3_results_baseline_v4.pkl.)')

print(f'\nNB6-baseline ({PIPELINE_VERSION}, multiclass, UNTUNED XGBoost) artifacts saved:')
for f in [f'attack3_results_baseline_{PIPELINE_VERSION}.pkl', f'adaptive_surrogate_baseline_{PIPELINE_VERSION}.pt',
          f'attack3_three_way_comparison_baseline_{PIPELINE_VERSION}.png']:
    if os.path.exists(f):
        print(f'  {f}  ({os.path.getsize(f)/1e6:.1f} MB)')
